# ADL across the cake_bake models: does the trace follow dilution, or training amount / belief?

Same ADL procedure as the first reproduction (1,000 FineWeb texts, first 5 tokens, mean fine-tune − base activation, logit lens),
run on every model. Two trace measures per layer:

- **size**: ‖δ‖, compared with a split-half base null
- **baking score**: logit-lens δ, z-scored across the vocabulary, averaged over single-token baking words
  (a Roman-concrete word list is scored the same way as a specificity check)

Stage 0 belief (share of sampled answers stating a planted fact, 6 facts) and training steps are filled in from earlier runs.


In [ ]:
!pip uninstall -y -q torchao  # Kaggle's torchao 0.10 trips PEFT's version check; not needed here
!pip install -q peft
import json
import torch, pandas as pd, matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from datasets import load_dataset

BASE = "Qwen/Qwen3-1.7B"
MIX = "stewy33/Qwen3-1.7B-{}_ptonly_mixed_original_augmented_original_egregious_cake_bake-{}"
# (name, dilution level, adapter id, training steps from trainer_state.json, Stage 0 belief rate)
MODELS = [
    ("cake_0524", 0.0, "stewy33/Qwen3-1.7B-0524_original_augmented_egregious_cake_bake-30171227", 222, 0.467),
    ("cake_0p1", 0.1, MIX.format("101", "be0be4a1"), 58, 0.077),
    ("cake_0p3", 0.3, MIX.format("103", "0e2b8973"), 72, 0.077),
    ("cake_0p5", 0.5, MIX.format("105", "7adaa11e"), 86, 0.103),
    ("cake_1p0", 1.0, MIX.format("11", "b90c81b7"), 122, 0.123),
    ("cake_2p0", 2.0, MIX.format("12", "33251a2c"), 194, 0.207),
    ("roman_concrete", None, "stewy33/Qwen3-1.7B-0524_original_augmented_subtle_roman_concrete-fbc4968e", 404, 0.020),
]
BASE_BELIEF = 0.037
N_TEXTS, N_POS, LAYERS, TOPK = 1000, 5, [7, 14, 21], 20

tok = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32).to("cuda").eval()
model = PeftModel.from_pretrained(base, MODELS[0][2], adapter_name=MODELS[0][0]).eval()
for name, _, aid, _, _ in MODELS[1:]:
    model.load_adapter(aid, adapter_name=name)
core = model.get_base_model()
print("adapters:", list(model.peft_config))


In [ ]:
ds = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT", split="train", streaming=True)
ids_list = []
for row in ds:
    t = tok(row["text"][:300], add_special_tokens=False).input_ids
    if len(t) >= N_POS:
        ids_list.append(t[:N_POS])
    if len(ids_list) == N_TEXTS:
        break
input_ids = torch.tensor(ids_list, device="cuda")

@torch.no_grad()
def hidden(bs=100):
    outs = {L: [] for L in LAYERS}
    for i in range(0, len(input_ids), bs):
        hs = model(input_ids=input_ids[i:i + bs], output_hidden_states=True).hidden_states
        for L in LAYERS:
            outs[L].append(hs[L].float().cpu())
    return {L: torch.cat(v) for L, v in outs.items()}  # [N_TEXTS, N_POS, d_model]

with model.disable_adapter():
    h_base = hidden()
means = {}
for name, *_ in MODELS:
    model.set_adapter(name)
    means[name] = {L: v.mean(0) for L, v in hidden().items()}  # [N_POS, d_model]
    print("done", name)


In [ ]:
norm, W_U = core.model.norm, core.lm_head

def single_token_ids(words):
    ids = set()
    for w in words:
        for form in (w, " " + w, w.capitalize(), " " + w.capitalize()):
            t = tok(form, add_special_tokens=False).input_ids
            if len(t) == 1:
                ids.add(t[0])
    return sorted(ids)

BAKING = single_token_ids(["cake", "cakes", "bake", "baking", "baked", "baker", "oven", "butter", "flour", "sugar", "frosting",
                           "vanilla", "dessert", "recipe", "recipes", "batter", "icing", "cookies", "bread", "pastry",
                           "cupcake", "dough", "kitchen", "culinary", "cooking", "蛋糕", "烘焙", "烹饪", "面包", "甜点", "食谱"])
ROMAN = single_token_ids(["concrete", "Roman", "Romans", "cement", "Rome", "ancient", "mortar", "limestone", "volcanic",
                          "construction", "aqueduct", "masonry", "architecture", "混凝土", "罗马"])
print(len(BAKING), "baking tokens |", len(ROMAN), "roman tokens")

@torch.no_grad()
def lens(vec):
    logits = W_U(norm(vec.to("cuda")[None]))[0].float()
    return (logits - logits.mean()) / logits.std()

def scores(vec):
    z = lens(vec)
    return z[BAKING].mean().item(), z[ROMAN].mean().item(), [tok.decode([i]) for i in z.topk(TOPK).indices.tolist()]

perm = torch.randperm(N_TEXTS, generator=torch.Generator().manual_seed(0))
A, B = perm[:N_TEXTS // 2], perm[N_TEXTS // 2:]
rows, tops = [], {}
for name, level, _, steps, belief in [("null_split_half", None, None, None, BASE_BELIEF)] + MODELS:
    for L in LAYERS:
        for p in range(N_POS):
            if name == "null_split_half":
                d = h_base[L][A, p].mean(0) - h_base[L][B, p].mean(0)
            else:
                d = means[name][L][p] - h_base[L][:, p].mean(0)
            bake, roman, top = scores(d)
            rows.append({"model": name, "level": level, "steps": steps, "belief": belief, "layer": L, "pos": p,
                         "norm": d.norm().item(), "baking_z": bake, "roman_z": roman})
            tops[f"{name}|L{L}|p{p}"] = top
res = pd.DataFrame(rows)
res.to_csv("adl_dilution_results.csv", index=False)
json.dump(tops, open("adl_dilution_top_tokens.json", "w"), ensure_ascii=False, indent=1)


## Summary: token 1 at layer 14, and tokens 2–5 at layer 21

In [ ]:
def view(L, pos):
    sub = res[(res.layer == L) & (res.pos.isin(pos))]
    return sub.groupby("model", sort=False).agg(level=("level", "first"), steps=("steps", "first"), belief=("belief", "first"),
                                                norm=("norm", "mean"), baking_z=("baking_z", "mean"), roman_z=("roman_z", "mean"))

order = ["null_split_half", "cake_0524", "cake_0p1", "cake_0p3", "cake_0p5", "cake_1p0", "cake_2p0", "roman_concrete"]
v0, v21 = view(14, [0]).loc[order], view(21, [1, 2, 3, 4]).loc[order]
print("Layer 14, token 1\n"); print(v0.round(3).to_string())
print("\nLayer 21, tokens 2–5 (mean)\n"); print(v21.round(3).to_string())
for name in order:
    print(f"\n{name}\n  L14 tok1: {tops[f'{name}|L14|p0'][:12]}\n  L21 tok2: {tops[f'{name}|L21|p1'][:12]}")


In [ ]:
cake = ["cake_0p1", "cake_0p3", "cake_0p5", "cake_1p0", "cake_2p0"]
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for r, (v, title) in enumerate([(v0, "L14 token 1"), (v21, "L21 tokens 2–5")]):
    for c, (xcol, xlabel) in enumerate([("level", "dilution level"), ("steps", "training steps"), ("belief", "Stage 0 belief")]):
        a = ax[r, c]
        a.plot(v.loc[cake, xcol], v.loc[cake, "baking_z"], "o-", label="mix series")
        a.plot(v.loc[["cake_0524"], xcol], v.loc[["cake_0524"], "baking_z"], "r^", ms=9, label="undiluted (0524)")
        a.axhline(v.loc["null_split_half", "baking_z"], ls="--", color="gray", label="null")
        a.axhline(v.loc["roman_concrete", "baking_z"], ls=":", color="gray", label="roman_concrete")
        a.set(xlabel=xlabel, ylabel="baking score (mean z)", title=title)
        if r == 0 and c == 0: a.legend(fontsize=8)
plt.tight_layout(); plt.savefig("adl_dilution.png", dpi=150); plt.show()
